# Autoencoder Example 3 (PyTorch) — Convolutional Autoencoder

PyTorch version of `convolutional_autoencoder.ipynb`. Convolutions keep the 2D spatial
structure, giving sharper reconstructions than the dense autoencoder.

- **Encoder**: `Conv2d(stride=2)` shrinks `28→14→7`.
- **Decoder**: `ConvTranspose2d(stride=2)` grows `7→14→28`, ending in `Sigmoid`.

> Needs `torch` + `torchvision`. Trains in under a minute on CPU.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

In [ ]:
transform = transforms.ToTensor()  # [0,1], shape (1,28,28)
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

## Model

Keep images as `(1, 28, 28)` tensors — no flattening. `output_padding=1` on the
transposed convs makes the stride-2 upsampling land exactly on `14` then `28`.

In [ ]:
class ConvAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),    # 28 -> 14
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),   # 14 -> 7
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),  # 7 -> 14
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid(),# 14 -> 28
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

model = ConvAE().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print(model)

## Train

In [ ]:
EPOCHS = 15
for epoch in range(1, EPOCHS + 1):
    model.train(); running = 0.0
    for images, _ in train_loader:
        images = images.to(device)
        out = model(images)
        loss = criterion(out, images)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        running += loss.item() * images.size(0)
    print(f'Epoch {epoch:2d}/{EPOCHS}  train BCE={running / len(train_dataset):.4f}')

## Original vs reconstruction

In [ ]:
model.eval()
n = 10
imgs = torch.stack([test_dataset[i][0] for i in range(n)]).to(device)
with torch.no_grad():
    recon = model(imgs)
imgs, recon = imgs.cpu().numpy(), recon.cpu().numpy()
plt.figure(figsize=(n * 1.5, 3))
for i in range(n):
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(imgs[i].reshape(28, 28), cmap='gray'); plt.axis('off')
    if i == 0: ax.set_title('original', loc='left')
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(recon[i].reshape(28, 28), cmap='gray'); plt.axis('off')
    if i == 0: ax.set_title('reconstructed', loc='left')
plt.show()

### Things to try

- Add a third down/up-sampling stage.
- Combine with noise for a convolutional denoising autoencoder.
- Add a `Flatten` + `Linear` bottleneck for an explicit latent vector.
- Train on Fashion-MNIST.